In [3]:
import numpy as np
from scipy.linalg import solve_banded


def derive_compact(u, direction, h):
    """6th-order compact derivative with free-slip walls (matches dery_11)."""
    alpha = 1/3; a = (7/9)/h; b = (1/36)/h
    n = u.shape[direction]

    # Tridiagonal matrix A in solve_banded format: [upper, diag, lower]
    ab = np.zeros((3, n))
    ab[0, 2:]      = alpha;   # upper
    ab[1, :]       = 1.0;     # diag
    ab[2, :-2]     = alpha;   # lower
    ab[1, 0]      += alpha**2  # free-slip corner BCs
    ab[1, -1]     += alpha**2

    # RHS: compact scheme stencil + wall corrections
    u_p1 = np.roll(u, -1, axis=direction)
    u_m1 = np.roll(u,  1, axis=direction)
    u_p2 = np.roll(u, -2, axis=direction)
    u_m2 = np.roll(u,  2, axis=direction)

    rhs = a * (u_p1 - u_m1) + b * (u_p2 - u_m2)

    # Free-slip ghost-point corrections at walls
    w0 = np.zeros_like(u); w0[tuple([slice(None)]*direction + [0] + [slice(None)]*(u.ndim-direction-1))] = 1
    wn = np.zeros_like(u); wn[tuple([slice(None)]*direction + [n-1] + [slice(None)]*(u.ndim-direction-1))] = 1

    rhs += a * (2*u_p1 - u_m1)*w0 + b * (2*u_p2 - u_m2)*w0   # bottom wall
    rhs += a * (u_p1 - 2*u_m1)*wn + b * (u_p2 - 2*u_m2)*wn   # top wall

    # Solve A @ du = rhs
    shape_after = np.roll(u.shape, -direction)
    rhs_r = np.moveaxis(rhs, direction, 0).reshape(n, -1)
    du_r  = solve_banded((1, 1), ab, rhs_r)
    return np.moveaxis(du_r.reshape(shape_after), 0, direction)


def rmse(X, Y):
    return np.sqrt(np.mean((X - Y) ** 2))
def rel_rmse(X, Y):
    return rmse(X, Y) / np.sqrt(np.mean(Y**2))

In [4]:
import numpy as np


def test_derive_compact():
    ny = 128
    y = np.linspace(-1, 1, ny)
    dy = y[1] - y[0]

    # f(y) = sin(πy), f'(y) = π*cos(πy)
    u = np.sin(np.pi * y)
    du_exact = np.pi * np.cos(np.pi * y)

    # Add z-dimension so it's 3D (like real usage)
    u_3d = u[np.newaxis, :, np.newaxis]  # (1, ny, 1)
    du_exact_3d = du_exact[np.newaxis, :, np.newaxis]

    du_computed = derive_compact(u_3d, direction=1, h=dy)

    print(f"Max relative error: {rel_rmse(du_computed[0,:,0], du_exact):.2e}")
    print(f"L2  relative error: {np.linalg.norm(du_computed - du_exact_3d) / np.linalg.norm(du_exact_3d):.2e}")

    # Check it also works on a 3D field with no trivial dims
    x = np.linspace(0, 2*np.pi, 64)
    z = np.linspace(0, 2*np.pi, 64)
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    u_xyz = np.sin(np.pi * Y) + 0.5*np.cos(X)*np.sin(Z)
    du_exact_xyz = np.pi * np.cos(np.pi * Y)

    du_computed_3d = derive_compact(u_xyz, direction=1, h=dy)
    print(f"Max rel error (full 3D): {rel_rmse(du_computed_3d, du_exact_xyz):.2e}")


test_derive_compact()


Max relative error: 2.50e-01
L2  relative error: 2.50e-01
Max rel error (full 3D): 7.41e-01
